# v14 — 제출 기회를 쓰지 않는 train 내부 오류 분석
하루 한 번의 제출 기회는 진단에 사용하지 않습니다. test/sample 접근은 차단하고 홀드아웃을 재평가하지 않습니다. 1차 +0.004·확인 평균 +0.003·각 분할 원본 비하락·노이즈 하락 0.003 이하를 사전 기준으로 삼았습니다. 기준 미통과는 추가 진단 결과로 뒤집지 않습니다. 검증된 캐시가 없으면 이 노트북에서 baseline/쌍 분류기를 재학습합니다.

In [1]:
import os,sys,re,json,pickle,cloudpickle,hashlib,platform,time,copy,warnings
from pathlib import Path
from collections import Counter
import numpy as np
import pandas as pd
from scipy import sparse
from scipy.special import softmax
from sklearn.base import BaseEstimator,TransformerMixin,clone
from sklearn.feature_extraction import DictVectorizer
from sklearn.feature_selection import chi2,f_classif
from sklearn.preprocessing import StandardScaler,MaxAbsScaler
from sklearn.multiclass import OneVsRestClassifier
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import ExtraTreesClassifier
from sklearn.metrics import f1_score,accuracy_score,classification_report,confusion_matrix
from lightgbm import LGBMClassifier
from threadpoolctl import threadpool_limits
from IPython.display import display,FileLink
threadpool_limits(2)
SEED=42
forbidden_attempts14=[]
def forbid_inference14(event,args):
 if event=='open' and args and isinstance(args[0],(str,bytes,os.PathLike)) and Path(os.fsdecode(args[0])).name in {'test.csv','sample_submission.csv'}:
  forbidden_attempts14.append(str(args[0]));raise RuntimeError('Inference data access is forbidden in v14')
sys.addaudithook(forbid_inference14)
roots=[Path.cwd(),*Path.cwd().parents]
REPO=next((p for p in roots if (p/'data/train.csv').is_file()),None)
if REPO is None:raise FileNotFoundError('data/train.csv가 필요합니다.')
DATA=REPO/'data';WORK=REPO/'models'/'혜진' if (REPO/'models'/'혜진').is_dir() else REPO
OUT=WORK/'artifacts'/'v14';OUT.mkdir(parents=True,exist_ok=True)
# Inference files and submission directory are intentionally not used.
os.environ['MPLCONFIGDIR']=str(OUT/'.mplconfig')
train=pd.read_csv(DATA/'train.csv')
genes=[c for c in train if c not in ['ID','SUBCLASS']]
assert train.ID.is_unique
y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);NC=len(labels)
X=train[genes];groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=SEED).split(X,y,groups))
folds=list(StratifiedGroupKFold(3,shuffle=True,random_state=SEED+1).split(X.iloc[dev],y[dev],groups[dev]))
assert not set(groups[dev]).intersection(groups[hold])
for a,b in folds:assert not set(groups[dev[a]]).intersection(groups[dev[b]])
pd.DataFrame({'ID':train.ID,'split':np.where(np.isin(np.arange(len(train)),hold),'holdout','development'),'group':groups.astype(str)}).to_csv(OUT/'split_manifest.csv',index=False)
print('TRAIN ONLY',train.shape,'Dev/holdout',len(dev),len(hold),'OUT',OUT,flush=True)

from sklearn.neural_network import MLPClassifier
from sklearn.feature_extraction.text import TfidfTransformer
from sklearn.utils.class_weight import compute_sample_weight
from xgboost import XGBClassifier


TRAIN ONLY (6201, 4386) Dev/holdout 4960 1241 OUT /Users/mac/project/cancer_project/artifacts/v14


In [2]:
GROUPS = {
 'p53_cellcycle': 'TP53 RB1 CDKN2A CDKN2B CDK4 CDK6 CCND1 ATM CHEK2'.split(),
 'rtk_ras_mapk': 'EGFR ERBB2 ERBB3 KRAS NRAS HRAS BRAF NF1 MAP2K1 MAP2K2 KIT PDGFRA FGFR2 FGFR3'.split(),
 'pi3k_mtor': 'PIK3CA PIK3R1 PTEN AKT1 MTOR TSC1 TSC2 STK11'.split(),
 'dna_repair': 'BRCA1 BRCA2 PALB2 ATM ATR CHEK2 MLH1 MSH2 MSH6 PMS2 POLE POLD1'.split(),
 'chromatin': 'ARID1A ARID1B ARID2 PBRM1 SMARCA4 BAP1 SETD2 KMT2A KMT2C KMT2D ATRX'.split(),
 'wnt': 'APC CTNNB1 AXIN1 AXIN2 RNF43'.split(),
 'metabolism': 'IDH1 IDH2 SDHB SDHC SDHD FH VHL'.split(),
}

def kind(token):
 if 'fs' in token.lower(): return 'frameshift'
 if '*' in token: return 'stop'
 if 'del' in token.lower(): return 'deletion'
 if 'ins' in token.lower(): return 'insertion'
 m = re.fullmatch(r'([A-Z])(\d+)([A-Z])', token)
 if m: return 'synonymous' if m[1] == m[3] else 'substitution'
 return 'other'

class MutationFeatures(BaseEstimator, TransformerMixin):
 def __init__(self, mode='full', min_df=3):
  self.mode = mode
  self.min_df = min_df

 def dictionaries(self, X):
  genes = list(X.columns)
  for row in X.itertuples(index=False, name=None):
   d = {}; counts = {}; burden = 0; events = 0; positions = []
   changed = set()
   for gene, value in zip(genes, row):
    if value is None or str(value) in ('nan', '', '<NA>'):
     d['missing:' + gene] = 1; continue
    if value == 'WT': continue
    changed.add(gene); burden += 1; d['gene:' + gene] = 1
    if self.mode == 'binary': continue
    tokens = str(value).split()
    for raw_token in tokens:
     token = re.sub(r'^([A-Z]\d+)X$', r'\1*', raw_token)
     events += 1; typ = kind(token); counts[typ] = counts.get(typ, 0) + 1
     d['variant:' + gene + ':' + token] = 1
     d['type:' + gene + ':' + typ] = 1
     m = re.search(r'\d+', token)
     if m:
      position = int(m[0]); positions.append(position)
      d['position:' + gene + ':' + str(position)] = 1
   if self.mode == 'full':
    d['numeric:log_mutated_genes'] = np.log1p(burden)
    d['numeric:log_events'] = np.log1p(events)
    d['numeric:multi_event_excess'] = np.log1p(max(0, events-burden))
    for typ in ['frameshift','stop','deletion','insertion','synonymous','substitution','other']:
     d['numeric:log_' + typ] = np.log1p(counts.get(typ,0))
     d['numeric:fraction_' + typ] = counts.get(typ,0)/max(events,1)
    d['numeric:log_mean_position'] = np.log1p(np.mean(positions)) if positions else 0.
    for name, members in GROUPS.items():
     d['module:' + name] = np.log1p(len(changed.intersection(members)))
   yield d

 def fit(self, X, y=None):
  self.genes_ = list(X.columns)
  self.vectorizer_ = DictVectorizer(dtype=np.float32)
  matrix = self.vectorizer_.fit_transform(self.dictionaries(X))
  names = self.vectorizer_.get_feature_names_out()
  self.keep_ = (np.asarray((matrix != 0).sum(axis=0)).ravel() >= self.min_df) | np.array([s.startswith(('numeric:', 'module:')) for s in names])
  self.feature_names_ = names[self.keep_]
  return self

 def transform(self, X):
  missing = set(self.genes_) - set(X.columns)
  if missing: raise ValueError('Missing gene columns: ' + str(sorted(missing)[:10]))
  return sparse.csr_matrix(self.vectorizer_.transform(self.dictionaries(X[self.genes_]))[:, self.keep_])

 def get_feature_names_out(self, input_features=None): return self.feature_names_
def canon(t):
 t=t.strip().removeprefix('p.');t=re.sub(r'(?<=\d)X$','*',t)
 return re.sub(r'^([A-Z]\d+)(?:[A-Z*]+)?fs.*$',r'\1fs',t)
def aa_change(t):
 m=re.fullmatch(r'([A-Z])\d+([A-Z*])',t)
 if m:return m[1]+'>'+m[2]
 m=re.match(r'([A-Z])\d+',t)
 if m:
  for suffix in ['fs','del','ins']:
   if suffix in t:return m[1]+'>'+suffix
 return None

def parse_events(frame):
 names=list(frame.columns);result=[]
 for row in frame.itertuples(index=False,name=None):
  result.append([(g,tuple(sorted(set(canon(t) for t in v.split())))) for g,v in zip(names,row) if isinstance(v,str) and v!='WT'])
 return result

class EvidenceEncoder:
 def __init__(self,unknown=1.,silent=1.,support=2):self.unknown=unknown;self.silent=silent;self.support=support
 def fit(self,events):
  self.token_df={};self.change_df={}
  if self.unknown<1:
   tokens=Counter();changes=Counter()
   for row in events:
    for g,ts in row:
     tokens.update((g,t) for t in ts)
     changes.update(set((g,a) for t in ts if (a:=aa_change(t)) is not None))
   self.token_df={k:v for k,v in tokens.items() if v>=self.support}
   self.change_df={k:v for k,v in changes.items() if v>=self.support}
  return self
 def transform(self,events,training=False,exclude_events=None):
  result=[]
  for row_number,row in enumerate(events):
   own=exclude_events[row_number] if exclude_events is not None else (row if training else [])
   own_tokens={(gg,tt) for gg,tts in own for tt in tts}
   own_changes={(gg,ac) for gg,tts in own for tt in tts if (ac:=aa_change(tt)) is not None}
   d={};gene_weights={};type_sums=Counter();non_syn=0.;trunc=0.;multikind=0
   for g,ts in row:
    types={};gene=0.;known_gene=False
    for t in ts:
     typ=kind(t);a=aa_change(t)
     known=self.token_df.get((g,t),0)-int((g,t) in own_tokens)>=self.support
     known_aa=a is not None and self.change_df.get((g,a),0)-int((g,a) in own_changes)>=self.support
     if typ=='synonymous':w=self.silent
     elif known or typ in ['stop','frameshift','deletion','insertion']:w=1.
     elif known_aa:w=max(self.unknown,.75)
     else:w=self.unknown
     if w<=0:continue
     gene=max(gene,w);types[typ]=max(types.get(typ,0),w)
     d['variant:'+g+':'+t]=max(d.get('variant:'+g+':'+t,0),w)
     m=re.search(r'\d+',t)
     if m:d['position:'+g+':'+m[0]]=max(d.get('position:'+g+':'+m[0],0),w)
     if a:d['aa:'+g+':'+a]=max(d.get('aa:'+g+':'+a,0),w)
     if known and typ!='synonymous':known_gene=True
    if gene<=0:continue
    d['gene:'+g]=gene;gene_weights[g]=gene
    for typ,w in types.items():d['type:'+g+':'+typ]=w;type_sums[typ]+=w
    if known_gene:d['supported:'+g]=1.
    non_syn+=max([w for k,w in types.items() if k!='synonymous'] or [0.])
    trunc+=max([w for k,w in types.items() if k in ['stop','frameshift','deletion','insertion']] or [0.])
    multikind+=len(types)>1
   n=len(row)
   d['numeric:log_raw_genes']=np.log1p(n)
   d['numeric:log_effective_genes']=np.log1p(sum(gene_weights.values()))
   d['numeric:log_nonsyn_genes']=np.log1p(non_syn)
   d['numeric:log_trunc_genes']=np.log1p(trunc)
   d['numeric:log_multikind_genes']=np.log1p(multikind)
   for typ in ['synonymous','substitution','stop','frameshift','deletion','insertion','other']:
    d['numeric:gene_fraction_'+typ]=type_sums[typ]/max(n,1)
   for name,gs in GROUPS.items():d['module:'+name]=np.log1p(sum(gene_weights.get(g,0) for g in gs))
   result.append(d)
  return result

events=parse_events(X)
legacy_records=list(MutationFeatures(mode='full').dictionaries(X))
assert canon('R213X')=='R213*' and canon('K16Rfs')=='K16fs'
assert aa_change('R132H')==aa_change('R140H')=='R>H'
print('Parsed rows:',len(events),'Normalization examples passed')


Parsed rows: 6201 Normalization examples passed


In [3]:
def relevance_scores(a,target,method):
 base=method.removesuffix('_full')
 if base=='embedded':
  scaled=csr32(MaxAbsScaler().fit_transform(a))
  fitted=OneVsRestClassifier(LogisticRegression(solver='liblinear',l1_ratio=1,C=.2,class_weight='balanced',max_iter=2000),n_jobs=1).fit(scaled,target)
  return np.max(np.abs(np.vstack([e.coef_[0] for e in fitted.estimators_])),axis=0)
 return (chi2(a,target) if base=='chi2' else f_classif(a,target))[0]

def csr32(a):
 b=sparse.csr_matrix(a,dtype=np.float32)
 if b.nnz>=np.iinfo(np.int32).max:raise ValueError('Sparse matrix too large for tree index type')
 b.indices=b.indices.astype(np.int32,copy=False);b.indptr=b.indptr.astype(np.int32,copy=False)
 return b

class Space:
 def fit(self,rs):
  self.vec=DictVectorizer(dtype=np.float32);a=self.vec.fit_transform(rs)
  names=self.vec.get_feature_names_out();df=np.asarray((a!=0).sum(axis=0)).ravel()
  keep=(df>=3)|np.array([n.startswith(('numeric:','module:')) for n in names])
  self.raw=np.flatnonzero(keep);self.names=names[keep]
  return csr32(a[:,self.raw])
 def transform(self,rs):return csr32(self.vec.transform(rs)[:,self.raw])
 def compact_vec(self,indices):
  v=copy.deepcopy(self.vec);mask=np.zeros(len(v.feature_names_),bool);mask[self.raw[np.sort(indices)]]=True
  v.restrict(mask);return v

class Selector:
 def __init__(self,a,target,names):
  self.a=a;self.y=target;self.names=names;self.cache={};self.corr={};self.scores={};self.audit=[];self.examples=[]
 def order(self,method,threshold):
  key=(method,threshold)
  if key in self.cache:return self.cache[key]
  if method not in self.scores:
   with warnings.catch_warnings():
    warnings.simplefilter('ignore',RuntimeWarning);score=relevance_scores(self.a,self.y,method)
   score=np.nan_to_num(score,nan=0,posinf=np.finfo(np.float32).max,neginf=0)
   if not method.endswith('_full'):score[np.array([n.startswith('numeric:') for n in self.names])]=0
   self.scores[method]=score
  score=self.scores[method]
  if method not in self.corr:
   pool=np.argsort(-score,kind='stable')[:min(4096,int((score>0).sum()))]
   z=self.a[:,pool];mean=np.asarray(z.mean(axis=0)).ravel();var=np.asarray(z.multiply(z).mean(axis=0)).ravel()-mean**2
   sd=np.sqrt(np.maximum(var,1e-12));corr=(z.T@z).toarray()/z.shape[0]-np.outer(mean,mean)
   corr/=np.outer(sd,sd);np.clip(corr,-1,1,out=corr)
   self.corr[method]=(pool,corr)
  pool,corr=self.corr[method];chosen=[]
  for i in range(len(pool)):
   if chosen:
    correlations=np.abs(corr[i,chosen]);j=int(np.argmax(correlations))
    if correlations[j]>=threshold:
     if len(self.examples)<100:self.examples.append({'method':method,'threshold':threshold,'kept':self.names[pool[chosen[j]]],'discarded':self.names[pool[i]],'correlation':float(corr[i,chosen[j]])})
     continue
   chosen.append(i)
   if len(chosen)>=2048:break
  order=pool[chosen];self.cache[key]=order
  self.audit.append({'method':method,'threshold':threshold,'association_pool':len(pool),'remaining_up_to_2048':len(order)})
  return order
 def select(self,method,threshold,k):return np.sort(self.order(method,threshold)[:k])

class CompactPredictor:
 def __init__(self,space,cols,est):
  self.vec=space.compact_vec(cols);self.names=self.vec.get_feature_names_out();self.est=est
  self.direct_inputs=len(self.names);self.raw_inputs=len(self.names);self.kind='compact'
 def predict(self,rs):return self.est.predict_proba(csr32(self.vec.transform(rs)))

def background_frame(frame,training_frame,seed,count):
 rng=np.random.default_rng(seed);out=frame.copy();arr=out.to_numpy(copy=True)
 rate=(training_frame.notna()&training_frame.ne('WT')).mean().to_numpy();prob=rate/rate.sum();idx=np.flatnonzero(prob>0)
 for r in range(len(arr)):
  n=min(len(idx),int(rng.poisson(count)))
  for c in rng.choice(len(prob),size=n,replace=False,p=prob):
   token='A1A' if rng.random()<.5 else 'A1V'
   arr[r,c]=token if not isinstance(arr[r,c],str) or arr[r,c]=='WT' else arr[r,c]+' '+token
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)
class ColumnPost:
 def __init__(self,mode='none'):self.mode=mode
 def fit(self,a,names):
  self.raw_names=np.array(names);self.names=np.array(names)
  if self.mode=='maxabs':self.scaler=MaxAbsScaler().fit(a)
  if self.mode=='block':
   self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names])
   self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
   self.tf=TfidfTransformer(norm='l2',use_idf=True).fit(a[:,self.binary])
   self.scaler=MaxAbsScaler().fit(a[:,self.summary])
   self.names=np.concatenate([self.raw_names[self.binary],self.raw_names[self.summary]])
  return self
 def transform(self,a):
  if self.mode=='maxabs':return csr32(self.scaler.transform(a))
  if self.mode=='block':return csr32(sparse.hstack([self.tf.transform(a[:,self.binary]),self.scaler.transform(a[:,self.summary])],format='csr'))
  return csr32(a)

class Member:
 def __init__(self,name,view,encoder,space,cols,post,est):
  self.name=name;self.view=view;self.encoder=encoder;self.vec=space.compact_vec(cols);self.post=post;self.est=est
  self.raw_names=self.vec.get_feature_names_out();self.names=post.names
 def predict_rows(self,rs):return self.est.predict_proba(self.post.transform(csr32(self.vec.transform(rs))))

VIEWS={'raw':(1.,1.),'syn_down':(1.,.25),'gated':(.25,1.),'gated_syn_down':(.25,.25),'gated_nonsyn':(.25,0.)}
CONFIGS={}
for view in VIEWS:
 for k in [1024,2048]:
  for c in [.1,.5]:CONFIGS[f'{view}_lr_k{k}_C{c}']=dict(view=view,algo='lr',k=k,C=c,post='none',method='chi2_full')
  CONFIGS[f'{view}_lgb_k{k}']=dict(view=view,algo='lgb',k=k,post='none',method='f_classif_full')
for view in ['raw','gated_syn_down']:
 for depth in [3,5]:CONFIGS[f'{view}_xgb_d{depth}']=dict(view=view,algo='xgb',k=2048,depth=depth,post='none',method='f_classif_full')
 for alpha in [1,10]:CONFIGS[f'{view}_mlp_a{alpha}']=dict(view=view,algo='mlp',k=1024,alpha=alpha,post='maxabs',method='chi2_full')
 for c in [1,4]:CONFIGS[f'{view}_blocklr_C{c}']=dict(view=view,algo='lr',k=2048,C=c,post='block',method='chi2_full')
 CONFIGS[f'{view}_extra']=dict(view=view,algo='extra',k=1024,post='none',method='f_classif_full')
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))

def estimator(cfg):
 algo=cfg['algo']
 if algo=='lr':return LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000)
 if algo=='lgb':return LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
 if algo=='xgb':return XGBClassifier(n_estimators=300,max_depth=cfg['depth'],learning_rate=.04,min_child_weight=3,reg_lambda=10,subsample=.85,colsample_bytree=.85,max_bin=63,tree_method='hist',objective='multi:softprob',eval_metric='mlogloss',n_jobs=2,random_state=SEED)
 if algo=='mlp':return MLPClassifier(hidden_layer_sizes=(128,),activation='relu',alpha=cfg['alpha'],learning_rate_init=.001,batch_size=128,max_iter=250,tol=1e-4,n_iter_no_change=15,early_stopping=False,random_state=SEED)
 return ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)

def prepare_view(idx,view):
 ev=[events[i] for i in idx];enc=EvidenceEncoder(*VIEWS[view]).fit(ev)
 rs=enc.transform(ev,training=True);space=Space();a=space.fit(rs);selector=Selector(a,yi[idx],space.names)
 return enc,space,a,selector

def train_new(name,idx,cache):
 cfg=CONFIGS[name];view=cfg['view']
 if view not in cache:cache[view]=prepare_view(idx,view)
 enc,space,a,selector=cache[view];cols=selector.select(cfg['method'],.98,cfg['k'])
 post=ColumnPost(cfg['post']).fit(a[:,cols],space.names[cols]);tx=post.transform(a[:,cols]);target=yi[idx]
 row_weights=np.ones(len(idx),dtype=float)
 if cfg.get('augment',False):
  if '_aug_events' not in cache:
   cache['_aug_events']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  aug_rs=enc.transform(cache['_aug_events'],exclude_events=[events[i] for i in idx])
  aug_x=post.transform(space.transform(aug_rs)[:,cols]);tx=csr32(sparse.vstack([tx,aug_x],format='csr'))
  target=np.concatenate([target,target]);row_weights=np.concatenate([row_weights,.5*row_weights])
 est=estimator(cfg)
 if cfg['algo'] in ['xgb','mlp']:est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*row_weights)
 elif cfg.get('augment',False):est.fit(tx,target,sample_weight=row_weights)
 else:est.fit(tx,target)
 return Member(name,view,enc,space,cols,post,est)

LEGACY={
 'legacy_lr':dict(k=2048,method='chi2',algo='lr'),
 'legacy_lgb':dict(k=2048,method='f_classif_full',algo='lgb'),
 'legacy_extra':dict(k=1024,method='f_classif_full',algo='extra')}
def train_legacy(idx):
 space=Space();a=space.fit([legacy_records[i] for i in idx]);sel=Selector(a,yi[idx],space.names);result=[]
 for name,cfg in LEGACY.items():
  cols=sel.select(cfg['method'],.95,cfg['k'])
  if cfg['algo']=='lr':est=LogisticRegression(C=.1,class_weight='balanced',max_iter=2000)
  elif cfg['algo']=='lgb':est=LGBMClassifier(n_estimators=250,learning_rate=.05,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=SEED)
  else:est=ExtraTreesClassifier(n_estimators=250,max_depth=20,min_samples_leaf=3,max_features='sqrt',class_weight='balanced',n_jobs=2,random_state=SEED)
  est.fit(a[:,cols],yi[idx]);post=ColumnPost().fit(a[:,cols],space.names[cols]);result.append(Member(name,'legacy',None,space,cols,post,est))
 return result

def duplicate_frame(frame):
 arr=frame.to_numpy(copy=True)
 for i,row in enumerate(arr):
  for j,v in enumerate(row):
   if isinstance(v,str) and v!='WT':arr[i,j]=v+' '+v
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def rare_background(frame,tr_events,seed,count=20):
 rng=np.random.default_rng(seed);df=Counter((g,t) for row in tr_events for g,ts in row for t in ts)
 syn=[];sub=[]
 for (g,t),n in df.items():
  if n>2:continue
  if kind(t)=='synonymous':syn.append((g,t))
  elif kind(t)=='substitution':sub.append((g,t))
 arr=frame.to_numpy(copy=True);lookup={g:i for i,g in enumerate(frame.columns)}
 for row in arr:
  for _ in range(int(rng.poisson(count))):
   pool=syn if rng.random()<.5 else sub
   g,t=pool[int(rng.integers(len(pool)))];j=lookup[g]
   row[j]=t if not isinstance(row[j],str) or row[j]=='WT' else row[j]+' '+t
 return pd.DataFrame(arr,columns=frame.columns,index=frame.index)

def eval_views(idx,tr_idx,seed):
 clean=X.iloc[idx];noise=rare_background(clean,[events[i] for i in tr_idx],seed);dup=duplicate_frame(clean)
 frames=[clean,noise,dup]
 return frames,[parse_events(f) for f in frames],[list(MutationFeatures(mode='full').dictionaries(f)) for f in frames]

def predict_members(members,weights,frame):
 ev=None;old=None;cache={};p=None
 for m,w in zip(members,weights):
  if m.view=='legacy':
   if old is None:old=list(MutationFeatures(mode='full').dictionaries(frame[genes]))
   rs=old
  else:
   if ev is None:ev=parse_events(frame[genes])
   if m.view not in cache:cache[m.view]=m.encoder.transform(ev)
   rs=cache[m.view]
  q=m.predict_rows(rs);p=q*w if p is None else p+q*w
 return p
print('Novel settings:',len(CONFIGS),'Views:',VIEWS)

# CV-guided targeted refinement: regularization, support threshold, and rare-variant augmentation.
VIEWS['gated_syn_down_s3']=(.25,.25,3)
CONFIGS.update({'raw_mlp_a0.01': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'raw_mlp_a0.1': {'view': 'raw', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.01': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.01, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_mlp_a0.1': {'view': 'gated_syn_down', 'algo': 'mlp', 'k': 1024, 'alpha': 0.1, 'post': 'maxabs', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k1024_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 1024, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k1024': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 1024, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_lr_k2048_C0.1': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.1, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lr_k2048_C0.5': {'view': 'gated_syn_down_s3', 'algo': 'lr', 'k': 2048, 'C': 0.5, 'post': 'none', 'method': 'chi2_full'}, 'gated_syn_down_s3_lgb_k2048': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d3': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 3, 'post': 'none', 'method': 'f_classif_full'}, 'gated_syn_down_s3_xgb_d5': {'view': 'gated_syn_down_s3', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full'}, 'raw_aug_lgb': {'view': 'raw', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'raw_aug_xgb_d5': {'view': 'raw', 'algo': 'xgb', 'k': 2048, 'depth': 5, 'post': 'none', 'method': 'f_classif_full', 'augment': True}, 'gated_s3_aug_lgb': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 2048, 'post': 'none', 'method': 'f_classif_full', 'augment': True}})
(OUT/'configurations.json').write_text(json.dumps(CONFIGS,indent=2))
print('Total new configurations after refinement:',len(CONFIGS))


Novel settings: 44 Views: {'raw': (1.0, 1.0), 'syn_down': (1.0, 0.25), 'gated': (0.25, 1.0), 'gated_syn_down': (0.25, 0.25), 'gated_nonsyn': (0.25, 0.0)}
Total new configurations after refinement: 59


In [4]:
# Freeze the previous version; do not repeat its 59-setting search.
BASE_CONFIGS=copy.deepcopy(CONFIGS)
BASE_PLAN=[('v4_reference',.25),('gated_s3_aug_lgb',.1875),('raw_aug_xgb_d5',.1875),('raw_lr_k2048_C0.1',.1875),('gated_syn_down_mlp_a1',.1875)]

def fit_base(idx):
 cache={};ms=[];ws=[]
 for name,w in BASE_PLAN:
  if name=='v4_reference':ms.extend(train_legacy(idx));ws.extend([w/3]*3)
  else:ms.append(train_new(name,idx,cache));ws.append(w)
 return ms,np.array(ws)


In [5]:
class NewSelector:
 def __init__(self,a,target,names):self.a=a;self.y=target;self.names=names;self.cache={}
 def select(self,method,k):
  if (method,k) in self.cache:return self.cache[(method,k)]
  numeric=np.array([s.startswith(('numeric:','module:')) for s in self.names])
  with warnings.catch_warnings():
   warnings.simplefilter('ignore',RuntimeWarning)
   global_score=np.nan_to_num(f_classif(self.a,self.y)[0],nan=0,posinf=0,neginf=0)
   if method=='balanced':
    orders=[]
    for c in range(NC):
     yes=self.y==c
     score=np.nan_to_num(chi2(self.a,yes)[0],nan=0,posinf=0,neginf=0)
     high=np.asarray(self.a[yes].mean(axis=0)).ravel()>np.asarray(self.a[~yes].mean(axis=0)).ravel()
     score[~high|numeric]=0
     orders.append(np.argsort(-score,kind='stable')[:min(k,int((score>0).sum()))])
    order=[]
    for rank in range(max(map(len,orders))):
     for o in orders:
      if rank<len(o):order.append(int(o[rank]))
    order+=list(np.argsort(-global_score,kind='stable'))
   else:order=list(np.argsort(-global_score,kind='stable'))
  z=self.a.tocsc();chosen=list(np.flatnonzero(numeric));seen=set(chosen);signatures=set()
  for j in order:
   j=int(j)
   if j in seen:continue
   seen.add(j)
   if global_score[j]<=0:continue
   lo,hi=z.indptr[j:j+2]
   sig=hashlib.sha256(z.indices[lo:hi].tobytes()+z.data[lo:hi].tobytes()).digest()
   if sig in signatures:continue
   signatures.add(sig);chosen.append(j)
   if len(chosen)>=k:break
  self.cache[(method,k)]=np.sort(chosen);return self.cache[(method,k)]

class SpecificityPost:
 def fit(self,a,names,target):
  self.names=np.array(names);self.binary=np.flatnonzero([not n.startswith(('numeric:','module:')) for n in names]);self.summary=np.flatnonzero([n.startswith(('numeric:','module:')) for n in names])
  z=(a[:,self.binary]!=0).astype(np.float32)
  counts=np.vstack([np.asarray(z[target==c].sum(axis=0)).ravel() for c in range(NC)])
  prior=np.bincount(target,minlength=NC)/len(target);freq=counts.sum(axis=0)
  posterior=(counts+10*prior[:,None])/(freq+10)
  kl=(posterior*np.log(np.maximum(posterior,1e-12)/prior[:,None])).sum(axis=0)
  self.weights=(.25+np.sqrt(np.maximum(kl,0))*freq/(freq+20)).astype(np.float32)
  self.scaler=MaxAbsScaler().fit(a[:,self.summary]);return self
 def transform(self,a):
  from sklearn.preprocessing import normalize
  block=normalize(a[:,self.binary].multiply(self.weights),norm='l2')
  return csr32(sparse.hstack([block,self.scaler.transform(a[:,self.summary])],format='csr'))

NEW={}
for view in ['raw','gated_syn_down_s3']:
 for method in ['global','balanced']:
  NEW[f'{view}_{method}_lgb4096']=dict(view=view,algo='lgb',k=4096,method=method)
for c in [.1,.5]:NEW[f'balanced_lr_C{c}']=dict(view='raw',algo='lr',k=2048,method='balanced',C=c)
for c in [1.,4.]:NEW[f'specificity_lr_C{c}']=dict(view='raw',algo='lr',k=4096,method='balanced',C=c,specificity=True)

def fit_new6(name,idx,cache):
 cfg=NEW[name];view=cfg['view']
 if view not in cache:
  enc,space,a,_=prepare_view(idx,view);cache[view]=(enc,space,a,NewSelector(a,yi[idx],space.names))
 enc,space,a,sel=cache[view];cols=sel.select(cfg['method'],cfg['k']);target=yi[idx]
 if cfg.get('specificity'):post=SpecificityPost().fit(a[:,cols],space.names[cols],target)
 else:post=ColumnPost().fit(a[:,cols],space.names[cols])
 est=estimator(cfg);tx=post.transform(a[:,cols])
 if cfg.get('augment'):
  if '_aug' not in cache:cache['_aug']=parse_events(rare_background(X.iloc[idx],[events[i] for i in idx],SEED+777,count=10))
  ar=enc.transform(cache['_aug'],exclude_events=[events[i] for i in idx]);ax=post.transform(space.transform(ar)[:,cols])
  tx=csr32(sparse.vstack([tx,ax],format='csr'));target=np.concatenate([target,target]);rw=np.r_[np.ones(len(idx)),np.full(len(idx),.5)]
 else:rw=np.ones(len(idx))
 if cfg['algo']=='xgb':est.fit(tx,target,sample_weight=compute_sample_weight('balanced',target)*rw)
 elif cfg.get('augment'):est.fit(tx,target,sample_weight=rw)
 else:est.fit(tx,target)
 m=Member(name,view,enc,space,cols,post,est)
 # Column order of a specificity block changes, but raw_names always describe the vectorizer input.
 if cfg.get('specificity'):m.names=np.concatenate([m.raw_names[post.binary],m.raw_names[post.summary]])
 return m
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))
print('New configurations',len(NEW))

NEW.update({'gated_s3_aug_global_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'global', 'augment': True}, 'gated_s3_aug_balanced_lgb4096': {'view': 'gated_syn_down_s3', 'algo': 'lgb', 'k': 4096, 'method': 'balanced', 'augment': True}, 'raw_aug_global_xgb4096_d3': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 3, 'augment': True}, 'raw_aug_global_xgb4096_d5': {'view': 'raw', 'algo': 'xgb', 'k': 4096, 'method': 'global', 'depth': 5, 'augment': True}})
(OUT/'new_configurations.json').write_text(json.dumps(NEW,indent=2))

print('Total new configurations including augmentation extension:',len(NEW))


New configurations 8
Total new configurations including augmentation extension: 12


In [6]:
def fit_plan6(idx,plan,basefit=None):
 ms=[];ws=[];cache={}
 for name,w in plan:
  if name=='v5_reference':
   old,ow=basefit if basefit is not None else fit_base(idx)
   ms.extend(old);ws.extend(ow*w)
  else:ms.append(fit_new6(name,idx,cache));ws.append(w)
 return ms,np.array(ws)

V6_PLAN=[('v5_reference',.75),('gated_s3_aug_global_lgb4096',.25)]
def fit_v6(idx):return fit_plan6(idx,V6_PLAN)


In [7]:
from sklearn.svm import LinearSVC
from sklearn.naive_bayes import ComplementNB
from sklearn.feature_selection import SelectKBest

def expand10(raw):
 t=canon(raw);m=re.fullmatch(r'([0-9]+)_([0-9]+)([A-Z]+)>([A-Z]+)',t)
 if m and int(m[2])-int(m[1])+1==len(m[3])==len(m[4]):
  return tuple(f'{a}{int(m[1])+j}{b}' for j,(a,b) in enumerate(zip(m[3],m[4])))
 return (t,)

def kind10(t):
 if 'fs' in t:return 'frameshift'
 if t.endswith('*'):return 'stop'
 if 'delins' in t:return 'delins'
 if 'del' in t:return 'deletion'
 if 'ins' in t:return 'insertion'
 m=re.fullmatch(r'([A-Z])[0-9]+([A-Z])',t)
 if m:return 'synonymous' if m[1]==m[2] else 'substitution'
 return 'other'

def packet10(frame,ev=None):
 if ev is None:ev=parse_events(frame[genes])
 resolved=[[(g,tuple(sorted({s for t in ts for s in expand10(t)}))) for g,ts in row] for row in ev]
 missing=frame[genes].isna().to_numpy()
 return dict(events=resolved,missing=missing)

def records10(packet,representation,numeric=True):
 rs=[]
 for i,row in enumerate(packet['events']):
  d={};syn=nonsyn=structural=unparsed=0
  for g,ts in row:
   d['gene:'+g]=1.;types={kind10(t) for t in ts}
   sy='synonymous' in types;ns=bool(types.intersection({'substitution','stop','frameshift','deletion','insertion','delins'}));st=bool(types.intersection({'stop','frameshift','deletion','insertion','delins'}));ot='other' in types
   syn+=sy;nonsyn+=ns;structural+=st;unparsed+=ot
   if representation=='type':
    for typ in types:d['type:'+g+':'+typ]=1.
   elif representation=='functional':
    for flag,family in [(sy,'syn_gene'),(ns,'nonsyn_gene'),(st,'structural_gene'),(ot,'unparsed_gene')]:
     if flag:d[family+':'+g]=1.
  if numeric:
   d['numeric:log_observed_mutated_genes']=np.log1p(len(row));d['numeric:log_missing_genes']=np.log1p(int(packet['missing'][i].sum()))
   if representation!='gene':
    for title,value in [('syn',syn),('nonsyn',nonsyn),('structural',structural),('unparsed',unparsed)]:
     d['numeric:log_'+title+'_genes']=np.log1p(value);d['numeric:fraction_'+title+'_genes']=value/max(len(row),1)
  rs.append(d)
 return rs

def mask_packet10(packet,probability,seed):
 rng=np.random.default_rng(seed);out=[];missing=packet['missing'].copy();gpos={g:j for j,g in enumerate(genes)}
 for i,row in enumerate(packet['events']):
  keep=[]
  for g,ts in row:
   if rng.random()<probability:missing[i,gpos[g]]=True
   else:keep.append((g,ts))
  out.append(keep)
 return dict(events=out,missing=missing)

def select_cols10(a,target,k):
 if k is None or a.shape[1]<=k:return np.arange(a.shape[1])
 with warnings.catch_warnings():
  warnings.simplefilter('ignore',RuntimeWarning);scores=np.nan_to_num(chi2(a,target)[0],nan=0,posinf=0,neginf=0)
 return np.sort(np.argsort(-scores,kind='stable')[:k])

class MissingBernoulli10:
 def __init__(self,alpha=.1,pooled=False,fit_prior=False,k=None):self.alpha=alpha;self.pooled=pooled;self.fit_prior=fit_prior;self.k=k
 def fit(self,a,target,missing,sample_weight=None):
  self.cols=select_cols10(a,target,self.k);a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols]);w=np.ones(len(target)) if sample_weight is None else np.asarray(sample_weight)
  self.classes_=np.unique(target);nclass=np.array([w[target==c].sum() for c in self.classes_])
  counts=np.vstack([np.asarray(a[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])
  absent=np.vstack([np.asarray(missing[target==c].multiply(w[target==c,None]).sum(axis=0)).ravel() for c in self.classes_])
  observed=nclass[:,None]-absent
  if self.pooled:
   denominator=w.sum()-np.asarray(missing.multiply(w[:,None]).sum(axis=0)).ravel()
   prior=np.clip(np.asarray(a.multiply(w[:,None]).sum(axis=0)).ravel()/np.maximum(denominator,1e-9),1e-7,1-1e-7)
   prob=(counts+self.alpha*prior)/(observed+self.alpha)
  else:prob=(counts+self.alpha)/(observed+2*self.alpha)
  prob=np.clip(prob,1e-7,1-1e-7);self.log0=np.log1p(-prob);self.logit=np.log(prob)-self.log0
  prior=nclass/nclass.sum() if self.fit_prior else np.full(len(nclass),1/len(nclass));self.logprior=np.log(prior)
  self.coef_=self.logit;return self
 def predict_proba(self,a,missing):
  a=csr32(a[:,self.cols]);missing=csr32(missing[:,self.cols])
  score=np.asarray(a@self.logit.T)+self.log0.sum(axis=1)+self.logprior-np.asarray(missing@self.log0.T)
  return softmax(score,axis=1)

class GroupSigmoidSVM10:
 def __init__(self,C=.1,k=2048,balanced_calibration=False):self.C=C;self.k=k;self.balanced_calibration=balanced_calibration
 def fit(self,a,target,profile_groups,sample_weight):
  self.classes_=np.unique(target);oof=np.zeros((len(target),len(self.classes_)),float)
  for tr,va in StratifiedGroupKFold(3,shuffle=True,random_state=310).split(a,target,profile_groups):
   assert not set(profile_groups[tr]).intersection(profile_groups[va])
   cols=select_cols10(a[tr],target[tr],self.k);scaler=MaxAbsScaler().fit(a[tr][:,cols])
   est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(scaler.transform(a[tr][:,cols])),target[tr],sample_weight=sample_weight[tr])
   scores=est.decision_function(csr32(scaler.transform(a[va][:,cols])))
   assert np.array_equal(est.classes_,self.classes_);oof[va]=scores
  self.calibrators=[]
  for j,c in enumerate(self.classes_):
   cal=LogisticRegression(C=1.,class_weight='balanced' if self.balanced_calibration else None,max_iter=1000).fit(oof[:,j,None],(target==c).astype(int),sample_weight=sample_weight);self.calibrators.append(cal)
  self.cols=select_cols10(a,target,self.k);self.scaler=MaxAbsScaler().fit(a[:,self.cols]);self.est=LinearSVC(C=self.C,class_weight='balanced',dual='auto',max_iter=5000,random_state=42).fit(csr32(self.scaler.transform(a[:,self.cols])),target,sample_weight=sample_weight);self.coef_=self.est.coef_;return self
 def predict_proba(self,a):
  score=self.est.decision_function(csr32(self.scaler.transform(a[:,self.cols])))
  p=np.column_stack([cal.predict_proba(score[:,j,None])[:,1] for j,cal in enumerate(self.calibrators)])
  return p/np.maximum(p.sum(axis=1,keepdims=True),1e-12)

class Member10:
 def __init__(self,name,space,est,representation,numeric,cols=None,scaler=None):
  self.name=name;self.view='v10_'+representation;self.representation=representation;self.numeric=numeric;self.vec=space.compact_vec(np.arange(len(space.names)));self.est=est;self.cols=est.cols if cols is None else cols;self.raw_names=space.names[self.cols];self.names=self.raw_names;self.scaler=scaler
  lookup={g:i for i,g in enumerate(genes)};self.feature_gene_positions=np.array([lookup.get(str(s).split(':')[1],-1) if ':' in s and not str(s).startswith('numeric:') else -1 for s in space.names])
 def missing_matrix(self,packet):
  out=np.zeros((len(packet['events']),len(self.feature_gene_positions)),bool);valid=self.feature_gene_positions>=0
  out[:,valid]=packet['missing'][:,self.feature_gene_positions[valid]]
  return csr32(sparse.csr_matrix(out))
 def predict_packet(self,packet,records=None):
  rs=records10(packet,self.representation,self.numeric) if records is None else records;a=csr32(self.vec.transform(rs))
  if isinstance(self.est,MissingBernoulli10):return self.est.predict_proba(a,self.missing_matrix(packet))
  if isinstance(self.est,GroupSigmoidSVM10):return self.est.predict_proba(a)
  tx=a[:,self.cols]
  if self.scaler is not None:tx=csr32(self.scaler.transform(tx))
  return self.est.predict_proba(tx)
 def predict_frame(self,frame):return self.predict_packet(packet10(frame[genes]))

predict_members6=predict_members
def predict_members(members,weights,frame):
 old=[(m,w) for m,w in zip(members,weights) if not hasattr(m,'predict_frame')];p=predict_members6([m for m,w in old],[w for m,w in old],frame) if old else None
 packet=None
 for m,w in zip(members,weights):
  if hasattr(m,'predict_frame'):
   if packet is None:packet=packet10(frame[genes])
   q=m.predict_packet(packet)*w;p=q if p is None else p+q
 return p

C10={
 'gene_nb_a0.1':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=False,k=None,mask=0.),
 'gene_nb_a1':dict(rep='gene',algo='nb',alpha=1.,pooled=False,fit_prior=False,k=None,mask=0.),
 'gene_nb_empirical_prior':dict(rep='gene',algo='nb',alpha=.1,pooled=False,fit_prior=True,k=None,mask=0.),
 'gene_nb_pooled20':dict(rep='gene',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=None,mask=0.),
 'type_nb_pooled20':dict(rep='type',algo='nb',alpha=20.,pooled=True,fit_prior=False,k=4096,mask=0.),
 'gene_lr':dict(rep='gene',algo='lr',C=.1,k=2048,mask=0.),
 'gene_lr_mask10':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.1),
 'gene_lr_mask25':dict(rep='gene',algo='lr',C=.1,k=2048,mask=.25),
 'type_lr':dict(rep='type',algo='lr',C=.1,k=2048,mask=0.),
 'type_lr_mask10':dict(rep='type',algo='lr',C=.1,k=2048,mask=.1),
 'functional_lr':dict(rep='functional',algo='lr',C=.1,k=2048,mask=0.),
 'functional_lr_mask10':dict(rep='functional',algo='lr',C=.1,k=2048,mask=.1),
 'gene_svm':dict(rep='gene',algo='svm',C=.1,k=2048,mask=0.),
 'type_svm':dict(rep='type',algo='svm',C=.1,k=2048,mask=0.),
 'functional_svm':dict(rep='functional',algo='svm',C=.1,k=2048,mask=0.),
 'functional_svm_mask10':dict(rep='functional',algo='svm',C=.1,k=2048,mask=.1),
 'functional_lgb':dict(rep='functional',algo='lgb',k=2048,mask=0.),
 'functional_lgb_mask10':dict(rep='functional',algo='lgb',k=2048,mask=.1),
 'functional_lgb_mask25':dict(rep='functional',algo='lgb',k=2048,mask=.25)}

def prepare10(idx,representation,numeric,cache):
 key=(representation,numeric)
 if key not in cache:
  if '_clean' not in cache:cache['_clean']=packet10(X.iloc[idx],[events[i] for i in idx])
  rs=records10(cache['_clean'],representation,numeric);space=Space();a=space.fit(rs);cache[key]=(space,a)
 return cache[key]

def train10(name,idx,cache):
 cfg=C10[name];numeric=cfg['algo'] not in ['nb','cnb'];space,a=prepare10(idx,cfg['rep'],numeric,cache);target=yi[idx];rw=np.ones(len(idx));train_groups=groups[idx]
 augmented=None
 if cfg['mask']:
  key=('masked',cfg['mask'])
  if key not in cache:cache[key]=mask_packet10(cache['_clean'],cfg['mask'],1042+int(cfg['mask']*100))
  augmented=cache[key];ar=records10(augmented,cfg['rep'],numeric);a_aug=space.transform(ar);a=csr32(sparse.vstack([a,a_aug],format='csr'));target=np.r_[target,target];train_groups=np.r_[train_groups,train_groups];rw=np.r_[rw,.5*rw]/1.5
 if cfg['algo']=='nb':
  est=MissingBernoulli10(cfg['alpha'],cfg['pooled'],cfg['fit_prior'],cfg['k'])
  temp=Member10(name,space,est,cfg['rep'],False,cols=np.arange(a.shape[1]));missing=temp.missing_matrix(cache['_clean']);est.fit(a,target,missing,sample_weight=rw)
  return Member10(name,space,est,cfg['rep'],False)
 if cfg['algo']=='cnb':
  cols=select_cols10(a,target,cfg['k']);est=ComplementNB(alpha=cfg['alpha'],norm=False).fit(a[:,cols],target,sample_weight=rw)
  return Member10(name,space,est,cfg['rep'],False,cols=cols)
 if cfg['algo']=='svm':
  est=GroupSigmoidSVM10(cfg['C'],cfg['k'],cfg.get('balanced_calibration',False)).fit(a,target,train_groups,rw);return Member10(name,space,est,cfg['rep'],True)
 # Outer selection and scaler are fit on the original training patients only.
 cols=select_cols10(prepare10(idx,cfg['rep'],numeric,cache)[1],yi[idx],cfg['k'])
 scaler=MaxAbsScaler().fit(prepare10(idx,cfg['rep'],numeric,cache)[1][:,cols]) if cfg['algo']=='lr' else None
 tx=a[:,cols];tx=csr32(scaler.transform(tx)) if scaler is not None else csr32(tx)
 est=LogisticRegression(C=cfg['C'],class_weight='balanced',max_iter=2000) if cfg['algo']=='lr' else LGBMClassifier(n_estimators=300,learning_rate=.04,num_leaves=15,min_child_samples=20,reg_lambda=10,colsample_bytree=.85,class_weight='balanced',verbosity=-1,n_jobs=2,random_state=42)
 est.fit(tx,target,sample_weight=rw);return Member10(name,space,est,cfg['rep'],True,cols=cols,scaler=scaler)

# A missing feature must contribute no Bernoulli presence or absence evidence.
toy=sparse.csr_matrix([[1,0],[1,0],[0,1],[0,1]],dtype=np.float32);toy_m=sparse.csr_matrix(toy.shape,dtype=np.float32)
nbtoy=MissingBernoulli10(alpha=1.).fit(toy,np.array([0,0,1,1]),toy_m)
assert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[1,0]],dtype=np.float32),sparse.csr_matrix([[0,0]],dtype=np.float32)),[[.9,.1]])
assert np.allclose(nbtoy.predict_proba(sparse.csr_matrix([[0,0]],dtype=np.float32),sparse.csr_matrix([[1,1]],dtype=np.float32)),[[.5,.5]])
assert all(kind10(t)=='synonymous' for t in expand10('1_2FS>FS'))
(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2));print('V10 candidates:',len(C10),'semantic checks passed')

C10.update({'functional_svm_balanced': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.0, 'balanced_calibration': True}, 'functional_svm_balanced_mask10': {'rep': 'functional', 'algo': 'svm', 'C': 0.1, 'k': 2048, 'mask': 0.1, 'balanced_calibration': True}, 'type_complement_nb_a0.1': {'rep': 'type', 'algo': 'cnb', 'alpha': 0.1, 'k': 4096, 'mask': 0.0}, 'type_complement_nb_a1': {'rep': 'type', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}, 'functional_complement_nb_a1': {'rep': 'functional', 'algo': 'cnb', 'alpha': 1.0, 'k': 4096, 'mask': 0.0}})
(OUT/'configurations_v10.json').write_text(json.dumps(C10,indent=2))
print('Extended candidates:',len(C10))


V10 candidates: 19 semantic checks passed
Extended candidates: 24


In [8]:
import pandas as pd,numpy as np,re,json,hashlib
from pathlib import Path
from sklearn.model_selection import StratifiedGroupKFold
r=REPO;t=pd.read_csv(r/'data/train.csv');x=t.drop(columns=['ID','SUBCLASS']);raw=pd.util.hash_pandas_object(x,index=False).to_numpy()
def canon(v):
 v=v.strip().removeprefix('p.');v=re.sub(r'(?<=\d)X$','*',v);return re.sub(r'^([A-Z]\d+)(?:[A-Z*]+)?fs.*$',r'\1fs',v)
z=x.fillna('__MISSING__').astype(str).copy()
for g in z:
 mask=~z[g].isin(['WT','__MISSING__'])
 if mask.any():z.loc[mask,g]=z.loc[mask,g].map(lambda s:' '.join(sorted(set(canon(v) for v in s.split()))))
c=pd.util.hash_pandas_object(z,index=False).to_numpy();y=t.SUBCLASS.to_numpy();dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=42).split(x,y,raw));audit={'raw_unique_profiles':len(np.unique(raw)),'canonical_unique_profiles':len(np.unique(c)),'holdout_canonical_overlap':len(set(c[dev])&set(c[hold])),'seeds':{}}
for seed in [43,109,211]:
 splits=list(StratifiedGroupKFold(3,shuffle=True,random_state=seed).split(x.iloc[dev],y[dev],raw[dev]));audit['seeds'][str(seed)]=[len(set(c[dev[a]])&set(c[dev[b]])) for a,b in splits]
(OUT/'canonical_group_audit.json').write_text(json.dumps(audit,indent=2));np.savez_compressed(OUT/'canonical_groups.npz',groups=c,dev=dev,hold=hold);print(json.dumps(audit),flush=True)


{"raw_unique_profiles": 5636, "canonical_unique_profiles": 5636, "holdout_canonical_overlap": 0, "seeds": {"43": [0, 0, 0], "109": [0, 0, 0], "211": [0, 0, 0]}}


## A. Fold 학습 빈도에 의한 작은 확률 보정
원본 환자별 비중이나 예측 개수는 조정하지 않습니다. 학습 부분의 class prior만 사용합니다. 기존 재분할도 같은 개발 환자이므로 독립 외부 검증이 아닙니다.

In [10]:
from pathlib import Path
import sys,os,re,json,hashlib,ast,time
import numpy as np,pandas as pd,nbformat as nf
from sklearn.model_selection import StratifiedGroupKFold
from sklearn.metrics import f1_score,classification_report,confusion_matrix
R=REPO;O=OUT
attempts=forbidden_attempts14
train=pd.read_csv(R/'data/train.csv');genes=[c for c in train if c not in ['ID','SUBCLASS']];X=train[genes];y=train.SUBCLASS.to_numpy();labels=np.sort(np.unique(y));yi=np.searchsorted(labels,y);groups=pd.util.hash_pandas_object(X,index=False).to_numpy()
dev,hold=next(StratifiedGroupKFold(5,shuffle=True,random_state=42).split(X,y,groups));assert not set(groups[dev]).intersection(groups[hold])
sha=hashlib.sha256((R/'data/train.csv').read_bytes()).hexdigest()
for version in [9,10,13]:
 proof_path=WORK/f'artifacts/v{version}/verification.json'
 if proof_path.exists():
  proof=json.loads(proof_path.read_text());assert sha==(proof.get('train_sha256') or proof['data_sha256']['train.csv'])
# Rules fixed before scoring any new candidate.
BETA=[0.,.05,.1,.15,.2,.3,.4]
protocol={'reference':'v10: 0.75*v6 + 0.25*ComplementNB','beta':BETA,'bases':['v6','v10'],'fit_rule':'(training class counts + 1), normalize, power beta; no fitting to validation labels','primary_seed':43,'confirmation_seeds':[109,211,313],'primary_gain_min':.004,'confirm_mean_gain_min':.003,'each_seed_clean_gain_floor':0.,'each_seed_noise_gain_floor':-.003,'holdout_policy':'never scored/read cached holdout scores in v14','test_policy':'never opened; audit hook','patient_gating':False,'submission_policy':'none; do not use daily submission for diagnosis'}
(O/'protocol.json').write_text(json.dumps(protocol,indent=2));print('Protocol frozen',flush=True)
# Validated train-only caches are optional; cache-free Run All refits both baselines.
R=REPO;O=OUT
train_sha14=hashlib.sha256((DATA/'train.csv').read_bytes()).hexdigest()
def validate_q14(q):
 assert np.array_equal(q['dev'],dev) and np.array_equal(q['labels'],labels)
def reference14(seed):
 path=OUT/f'reference_seed{seed}.npz'
 if path.exists():
  q=np.load(path,allow_pickle=True);validate_q14(q);assert str(q['train_sha256'])==train_sha14
  return {'v6':q['v6'],'v10':q['v10']}
 version10=WORK/'artifacts/v10';proof_path=version10/'verification.json'
 if proof_path.exists():
  proof=json.loads(proof_path.read_text());assert proof['data_sha256']['train.csv']==train_sha14
  cache=version10/('oof_probabilities.npz' if seed==43 else 'confirmation_probabilities.npz')
  if cache.exists():
   q=np.load(cache,allow_pickle=True);validate_q14(q)
   k6='v6_reference' if seed==43 else f's{seed}__v6_reference';kn='type_complement_nb_a0.1' if seed==43 else f's{seed}__type_complement_nb_a0.1'
   if k6 in q and kn in q:
    base={'v6':q[k6].copy(),'v10':.75*q[k6]+.25*q[kn]}
    for p in base.values():assert p.shape==(3,len(dev),NC) and np.isfinite(p).all() and np.allclose(p.sum(2),1,atol=1e-5)
    return base
 base={key:np.zeros((3,len(dev),NC),np.float32) for key in ['v6','v10']}
 for f,(a,b) in enumerate(splits[seed]):
  tr=dev[a];frames,_,_=eval_views(dev[b],tr,42+901+f if seed==43 else 42+seed*10+f);ms,ws=fit_v6(tr);nb=train10('type_complement_nb_a0.1',tr,{})
  for j,frame in enumerate(frames):base['v6'][j,b]=predict_members(ms,ws,frame);base['v10'][j,b]=.75*base['v6'][j,b]+.25*nb.predict_frame(frame)
  print('Recomputed reference',seed,f,flush=True)
 np.savez_compressed(path,train_sha256=train_sha14,dev=dev,labels=labels,**base);return base
splits={seed:list(StratifiedGroupKFold(3,shuffle=True,random_state=seed).split(X.iloc[dev],y[dev],groups[dev])) for seed in [43,109,211]}
Q={seed:reference14(seed) for seed in [43,109,211]}

rows=[];predictions={};priors={}
for seed in Q:
 for fold,(a,b) in enumerate(splits[seed]):
  counts=np.bincount(yi[dev[a]],minlength=len(labels))+1;prior=counts/counts.sum();priors[f'{seed}_{fold}']=prior.tolist()
  for base in ['v6','v10']:
   for beta in BETA:
    name=f'{base}_beta{beta}';v=[]
    for j in range(3):
     p=Q[seed][base][j,b]*prior[None,:]**beta;p/=p.sum(1,keepdims=True);v.append(f1_score(yi[dev[b]],p.argmax(1),average='macro',zero_division=0))
     if j==0:predictions[(seed,name,fold)]=p
    rows.append(dict(seed=seed,fold=fold,base=base,beta=beta,name=name,clean=v[0],noise=v[1],duplicate=v[2],score=.6*v[0]+.3*v[1]+.1*v[2]))
fold=pd.DataFrame(rows);fold.to_csv(O/'fold_comparison.csv',index=False);summary=fold.groupby(['seed','name','base','beta'])[['clean','noise','duplicate','score']].mean().reset_index();summary.to_csv(O/'seed_comparison.csv',index=False)
rank=summary[summary.seed==43].sort_values('clean',ascending=False);rank.to_csv(O/'primary_ranking.csv',index=False);base=rank[rank.name=='v10_beta0.0'].iloc[0];eligible=rank[(rank.clean>=base.clean+protocol['primary_gain_min'])&(rank.noise>=base.noise-.003)]
# Exactly one finalist frozen based on primary; never rerank by confirmation.
finalist=None if eligible.empty else eligible.iloc[0].to_dict();(O/'frozen_finalist.json').write_text(json.dumps(finalist,indent=2))
# Error analysis uses development OOF only, no holdout labels or scores.
for tag,pr in [('v6',Q[43]['v6'][0]),('v10',Q[43]['v10'][0])]:
 pred=pr.argmax(1);report=pd.DataFrame(classification_report(y[dev],labels[pred],output_dict=True,zero_division=0)).T;report.to_csv(O/f'{tag}_development_class_report.csv');cm=confusion_matrix(yi[dev],pred,labels=np.arange(len(labels)));pairs=[dict(true=labels[i],predicted=labels[j],count=int(cm[i,j])) for i in range(len(labels)) for j in range(len(labels)) if i!=j and cm[i,j]>0];pd.DataFrame(pairs).sort_values('count',ascending=False).to_csv(O/f'{tag}_development_confusions.csv',index=False)
result={'status':'screen_completed','finalist':finalist,'train_sha256':sha,'forbidden_input_attempts':attempts,'dev_n':len(dev),'holdout_scored':False,'test_opened':False,'priors':priors}
(O/'screen_result.json').write_text(json.dumps(result,indent=2));print(rank[['name','clean','noise','score']].to_string(index=False),flush=True);print('FINALIST',finalist,flush=True);print(summary[summary.seed!=43][['seed','name','clean','noise','score']].to_string(index=False),flush=True)


Protocol frozen
        name    clean    noise    score
 v10_beta0.0 0.442404 0.345993 0.413261
v10_beta0.05 0.441337 0.345714 0.412393
 v10_beta0.1 0.440589 0.348863 0.412604
v10_beta0.15 0.439863 0.349932 0.412854
 v10_beta0.2 0.438460 0.346548 0.410650
 v6_beta0.05 0.436745 0.345595 0.409100
 v10_beta0.3 0.436232 0.345100 0.408551
  v6_beta0.0 0.435576 0.345775 0.408408
  v6_beta0.1 0.435086 0.345568 0.407922
 v6_beta0.15 0.433806 0.348059 0.407699
 v10_beta0.4 0.432332 0.344160 0.405985
  v6_beta0.4 0.430871 0.347518 0.405616
  v6_beta0.3 0.429841 0.347324 0.404770
  v6_beta0.2 0.429371 0.349949 0.405393
FINALIST None
 seed         name    clean    noise    score
  109  v10_beta0.0 0.436537 0.326650 0.403848
  109 v10_beta0.05 0.436865 0.329678 0.404824
  109  v10_beta0.1 0.438172 0.332915 0.406960
  109 v10_beta0.15 0.439570 0.329836 0.407004
  109  v10_beta0.2 0.442361 0.330596 0.409011
  109  v10_beta0.3 0.439460 0.327101 0.405954
  109  v10_beta0.4 0.434906 0.324644 0.401955
  

## B. 개발 OOF 오류에 근거한 쌍별 이진 분류
빈도 보정 실패 후 추가한 적응 탐색입니다. 선별과 모델 학습은 해당 fold의 쌍 학습 환자로 한정합니다. 고정 alpha를 모든 환자에게 동일하게 적용합니다.

In [12]:
assert hashlib.sha256((DATA/'train.csv').read_bytes()).hexdigest()==json.loads((OUT/'screen_result.json').read_text())['train_sha256']
# Second train-only experiment after prior-tilt screen failed. Disclosed adaptive design.
PAIRS14=[('GBMLGG','LGG'),('KIPAN','KIRC'),('BRCA','OV'),('PRAD','PCPG')]
ALGO14=['lr_empirical','lr_balanced','xgb_d2','lgb_l3'];ALPHA14=[.25,.5]
PAIR_PROTOCOL14={'pairs':PAIRS14,'algorithms':ALGO14,'scopes':[2,4],'alpha':ALPHA14,'input_k':512,'criterion_primary_clean_gain_vs_v10':.004,'criterion_confirm_mean_gain':.003,'criterion_each_seed_clean_gain':0.,'criterion_each_seed_noise_floor':-.003,'holdout_evaluation':False,'test_access':False,'adaptive_reason':'prior-only correction failed; investigate development OOF pair confusions','targeted_patient_gating':False}
(OUT/'pair_protocol.json').write_text(json.dumps(PAIR_PROTOCOL14,indent=2))
class PairHead14:
 def __init__(self,pair,algo,space,cols,est,scaler):self.pair=pair;self.algo=algo;self.vec=space.compact_vec(cols);self.names=space.names[cols];self.est=est;self.scaler=scaler
 def predict_records(self,records):
  a=csr32(self.vec.transform(records));a=csr32(self.scaler.transform(a)) if self.scaler is not None else a
  return self.est.predict_proba(a)[:,1]
 def predict_frame(self,frame):return self.predict_records(EvidenceEncoder(1.,1.).fit([]).transform(parse_events(frame[genes])))
def fit_pair14(tr,pair,algo,cache):
 idx=tr[np.isin(y[tr],pair)];target=(y[idx]==pair[1]).astype(int);key=tuple(pair)
 if key not in cache:
  records=EvidenceEncoder(1.,1.).fit([]).transform([events[i] for i in idx]);space=Space();a=space.fit(records);cols=select_cols10(a,target,512);scaler=MaxAbsScaler().fit(a[:,cols]);cache[key]=(space,cols,scaler,csr32(a[:,cols]))
 space,cols,scaler,a=cache[key]
 if algo.startswith('lr'):
  est=LogisticRegression(C=.1,class_weight='balanced' if algo=='lr_balanced' else None,max_iter=1500).fit(csr32(scaler.transform(a)),target);sc=scaler
 elif algo=='xgb_d2':
  est=XGBClassifier(n_estimators=150,max_depth=2,learning_rate=.04,min_child_weight=5,reg_lambda=20,subsample=1.,colsample_bytree=.8,tree_method='hist',max_bin=64,n_jobs=2,random_state=42,eval_metric='logloss').fit(a,target);sc=None
 else:
  est=LGBMClassifier(n_estimators=120,num_leaves=3,min_child_samples=15,learning_rate=.04,reg_lambda=20,verbosity=-1,n_jobs=2,random_state=42).fit(a,target);sc=None
 return PairHead14(pair,algo,space,cols,est,sc)
def apply_pair14(p,conditional,pairs,alpha):
 out=p.copy()
 for j,pair in enumerate(pairs):
  left,right=[int(np.flatnonzero(labels==c)[0]) for c in pair];mass=p[:,left]+p[:,right];old=np.divide(p[:,right],mass,out=np.full(len(p),.5),where=mass>0);new=(1-alpha)*old+alpha*conditional[j];out[:,right]=mass*new;out[:,left]=mass*(1-new)
 assert np.allclose(out.sum(1),1,atol=1e-5);return out
pair_cache=OUT/'pair_conditional_oof.npz';pair_meta=OUT/'pair_cache_metadata.json'
if pair_cache.exists() and pair_meta.exists():
 cm=json.loads(pair_meta.read_text());assert cm['train_sha256']==sha and cm['protocol']==json.loads((OUT/'pair_protocol.json').read_text())
 qpair=np.load(pair_cache,allow_pickle=True);assert np.array_equal(qpair['dev'],dev) and np.array_equal(qpair['labels'],labels);PAIR14=qpair['conditional'];pair_audit14=pd.read_csv(OUT/'pair_feature_audit.csv').to_dict('records')
 print('Validated pair primary cache reused')
else:
 PAIR14=np.zeros((len(ALGO14),4,3,len(dev)),np.float32);pair_audit14=[]
 for fold,(a,b) in enumerate(splits[43]):
  tr=dev[a];va=dev[b];frames,evs,_=eval_views(va,tr,42+901+fold);records=[EvidenceEncoder(1.,1.).fit([]).transform(ev) for ev in evs];cache={}
  for ai,algo in enumerate(ALGO14):
   for j,pair in enumerate(PAIRS14):
    m=fit_pair14(tr,pair,algo,cache)
    for view in range(3):PAIR14[ai,j,view,b]=m.predict_records(records[view])
    pair_audit14.append(dict(fold=fold,algo=algo,pair='/'.join(pair),n_train=int(np.isin(y[tr],pair).sum()),features=len(m.names)))
   print('V14 pair heads primary',fold,algo,'complete',flush=True)
  np.savez_compressed(OUT/'pair_conditional_oof.npz',dev=dev,labels=labels,conditional=PAIR14)
 pair_meta.write_text(json.dumps({'train_sha256':sha,'protocol':json.loads((OUT/'pair_protocol.json').read_text())},indent=2))
rows14=[]
for ai,algo in enumerate(ALGO14):
 for scope in [2,4]:
  for alpha in ALPHA14:
   for fold,(a,b) in enumerate(splits[43]):
    vals=[]
    for view in range(3):
     p=apply_pair14(Q[43]['v10'][view,b],PAIR14[ai,:scope,view][:,b],PAIRS14[:scope],alpha);vals.append(f1_score(yi[dev[b]],p.argmax(1),average='macro',zero_division=0))
    rows14.append(dict(name=f'{algo}_scope{scope}_a{alpha}',algo=algo,scope=scope,alpha=alpha,fold=fold,clean=vals[0],noise=vals[1],duplicate=vals[2],score=.6*vals[0]+.3*vals[1]+.1*vals[2]))
pd.DataFrame(rows14).to_csv(OUT/'pair_primary_folds.csv',index=False);pd.DataFrame(pair_audit14).to_csv(OUT/'pair_feature_audit.csv',index=False);pair_rank14=pd.DataFrame(rows14).groupby(['name','algo','scope','alpha'])[['clean','noise','duplicate','score']].mean().reset_index().sort_values('clean',ascending=False);pair_rank14.to_csv(OUT/'pair_primary_ranking.csv',index=False)
base14=summary[(summary.seed==43)&(summary.name=='v10_beta0.0')].iloc[0];eligible14=pair_rank14[(pair_rank14.clean>=base14.clean+.004)&(pair_rank14.noise>=base14.noise-.003)];pair_finalist14=None if eligible14.empty else eligible14.iloc[0].to_dict();(OUT/'pair_frozen_finalist.json').write_text(json.dumps(pair_finalist14,indent=2));print('PAIR PRIMARY',pair_rank14.to_string(index=False),flush=True);print('PAIR FROZEN',pair_finalist14,flush=True)


Validated pair primary cache reused
PAIR PRIMARY                      name         algo  scope  alpha    clean    noise  duplicate    score
 lr_empirical_scope2_a0.5 lr_empirical      2   0.50 0.444642 0.343785   0.442550 0.414176
 lr_empirical_scope4_a0.5 lr_empirical      4   0.50 0.444358 0.341570   0.444440 0.413530
lr_empirical_scope4_a0.25 lr_empirical      4   0.25 0.444102 0.342785   0.443574 0.413654
lr_empirical_scope2_a0.25 lr_empirical      2   0.25 0.443656 0.343877   0.441441 0.413501
      lgb_l3_scope2_a0.25       lgb_l3      2   0.25 0.443544 0.345877   0.441976 0.414087
      xgb_d2_scope2_a0.25       xgb_d2      2   0.25 0.443526 0.345104   0.441509 0.413798
       lgb_l3_scope2_a0.5       lgb_l3      2   0.50 0.443477 0.337180   0.442806 0.411521
      xgb_d2_scope4_a0.25       xgb_d2      4   0.25 0.442940 0.343312   0.442425 0.413000
      lgb_l3_scope4_a0.25       lgb_l3      4   0.25 0.442821 0.344195   0.443190 0.413270
       lgb_l3_scope4_a0.5       lgb_l3   

## C. 고정 후보의 추가 진단
1차 기준 미통과 상태를 바꾸지 않는 진단입니다. 후보와 비중을 더 튜닝하지 않고 109/211/313 분할과 그룹 bootstrap으로 불확실성을 확인합니다.

In [14]:
diag_meta=OUT/'diagnostic_cache_metadata.json'
if diag_meta.exists() and (OUT/'diagnostic_probabilities.npz').exists() and (OUT/'bootstrap_diagnostic.json').exists():
 cm=json.loads(diag_meta.read_text());diagnostic14=pair_rank14.iloc[0].to_dict();assert cm['train_sha256']==sha and cm['candidate']==diagnostic14
 diag14=pd.read_csv(OUT/'diagnostic_repeated_cv.csv').to_dict('records');ci14=json.loads((OUT/'bootstrap_diagnostic.json').read_text());print('Validated fixed diagnostic cache reused')
else:
 # No primary candidate passed. This diagnostic cannot overturn the original adoption gate.
 diagnostic14=pair_rank14.iloc[0].to_dict();(OUT/'diagnostic_frozen_candidate.json').write_text(json.dumps({'candidate':diagnostic14,'cannot_override_primary_gate':True,'seeds':[109,211,313],'bootstrap_replicates':500,'bootstrap_seed':14014},indent=2))
 condition14={};diag14=[];mix14={};base_diag14={}
 # Reconstruct the canonical split used by v13 seed 313 and verify reference fold metrics exactly.
 audit14=json.loads((OUT/'canonical_group_audit.json').read_text());assert audit14['holdout_canonical_overlap']==0 and all(v==0 for vals in audit14['seeds'].values() for v in vals)
 grp14=np.load(OUT/'canonical_groups.npz')['groups'];splits[313]=list(StratifiedGroupKFold(3,shuffle=True,random_state=313).split(X.iloc[dev],y[dev],grp14[dev]))
 ref_path=WORK/'artifacts/v13/confirmation_probabilities.npz'
 if ref_path.exists():
  q13=np.load(WORK/'artifacts/v13/confirmation_probabilities.npz',allow_pickle=True);assert np.array_equal(q13['dev'],dev) and np.array_equal(q13['labels'],labels)
  check13=pd.read_csv(WORK/'artifacts/v13/confirmation_cv.csv')
  for f,(a,b) in enumerate(splits[313]):
   expected=float(check13[(check13.fold==f)&(check13.plan=='v6_reference')].macro_f1.iloc[0]);assert abs(f1_score(yi[dev[b]],q13['v6_reference'][b].argmax(1),average='macro')-expected)<1e-10
 else:
  ref=np.zeros((len(dev),NC),np.float32)
  for f,(a,b) in enumerate(splits[313]):
   ms,ws=fit_v6(dev[a]);ref[b]=predict_members(ms,ws,X.iloc[dev[b]])
  q13={'v6_reference':ref}
 Q[313]={'v6':np.zeros((3,len(dev),len(labels)),np.float32),'v10':np.zeros((3,len(dev),len(labels)),np.float32)};Q[313]['v6'][0]=q13['v6_reference'];Q[313]['v10'][0]=.75*q13['v6_reference']
 for seed in [109,211,313]:
  cond=np.zeros((int(diagnostic14['scope']),3,len(dev)),np.float32)
  for f,(a,b) in enumerate(splits[seed]):
   tr=dev[a];va=dev[b];frames,evs,_=eval_views(va,tr,42+seed*10+f);records=[EvidenceEncoder(1.,1.).fit([]).transform(ev) for ev in evs];cache={}
   if seed==313:
    nb=train10('type_complement_nb_a0.1',tr,{})
    Q[313]['v10'][0,b]+=.25*nb.predict_frame(frames[0])
    # Only clean 313 reference exists; do not fabricate stress baselines or refit unnecessary heavy models.
   for j,pair in enumerate(PAIRS14[:int(diagnostic14['scope'])]):
    m=fit_pair14(tr,pair,diagnostic14['algo'],cache)
    for view in range(3):cond[j,view,b]=m.predict_records(records[view])
   views=[0] if seed==313 else [0,1,2]
   for view in views:
    base=Q[seed]['v10'][view,b];new=apply_pair14(base,cond[:,view][:,b],PAIRS14[:int(diagnostic14['scope'])],diagnostic14['alpha']);oldf=f1_score(yi[va],base.argmax(1),average='macro',zero_division=0);newf=f1_score(yi[va],new.argmax(1),average='macro',zero_division=0);diag14.append(dict(seed=seed,fold=f,view=['clean','noise','duplicate'][view],v10_f1=oldf,candidate_f1=newf,delta_f1=newf-oldf))
   pd.DataFrame(diag14).to_csv(OUT/'diagnostic_repeated_cv.csv',index=False);print('V14 fixed diagnostic',seed,f,'complete',flush=True)
  condition14[seed]=cond;mix14[seed]=apply_pair14(Q[seed]['v10'][0],cond[:,0],PAIRS14[:int(diagnostic14['scope'])],diagnostic14['alpha']);base_diag14[seed]=Q[seed]['v10'][0]
 ai=ALGO14.index(diagnostic14['algo']);scope=int(diagnostic14['scope']);mix14[43]=apply_pair14(Q[43]['v10'][0],PAIR14[ai,:scope,0],PAIRS14[:scope],diagnostic14['alpha']);base_diag14[43]=Q[43]['v10'][0]
 np.savez_compressed(OUT/'diagnostic_probabilities.npz',dev=dev,labels=labels,**{f's{seed}_candidate':p for seed,p in mix14.items()},**{f's{seed}_v10':p for seed,p in base_diag14.items()})
 # Paired group bootstrap, conditional on already fitted OOF predictions. Not an independent replication.
 rng=np.random.default_rng(14014);ug,inv=np.unique(groups[dev],return_inverse=True);codes0=yi[dev]*len(labels)+base_diag14[43].argmax(1);codes1=yi[dev]*len(labels)+mix14[43].argmax(1)
 def macro_cm14(codes,weights):
  cm=np.bincount(codes,weights=weights,minlength=len(labels)**2).reshape(len(labels),len(labels));den=cm.sum(0)+cm.sum(1);return np.divide(2*np.diag(cm),den,out=np.zeros(len(labels)),where=den>0).mean()
 draw14=[]
 for i in range(500):
  weights=rng.multinomial(len(ug),np.full(len(ug),1/len(ug)))[inv];draw14.append(macro_cm14(codes1,weights)-macro_cm14(codes0,weights))
 ci14={'kind':'paired profile-group bootstrap conditional on fixed development OOF predictions','delta_pooled_macro_f1':float(macro_cm14(codes1,np.ones(len(dev)))-macro_cm14(codes0,np.ones(len(dev)))),'ci95':np.quantile(draw14,[.025,.975]).tolist(),'replicates':500,'selection_bias_not_corrected':True,'not_independent_patient_evidence':True};(OUT/'bootstrap_diagnostic.json').write_text(json.dumps(ci14,indent=2));print(pd.DataFrame(diag14).groupby(['seed','view'])[['v10_f1','candidate_f1','delta_f1']].mean().to_string(),flush=True);print('BOOTSTRAP',ci14,flush=True)
 diag_meta.write_text(json.dumps({'train_sha256':sha,'candidate':diagnostic14},indent=2))


Validated fixed diagnostic cache reused


## D. 보고서와 검증 증거 저장
미채택 실험에는 새 홀드아웃 점수나 대회 점수를 붙이지 않습니다.

In [16]:
# No candidate passed the primary +0.004 gate. Diagnostic findings cannot relax it.
assert pair_finalist14 is None and finalist is None
paired_summary14=pd.DataFrame(diag14).groupby(['seed','view'])[['v10_f1','candidate_f1','delta_f1']].mean()
ci14=json.loads((OUT/'bootstrap_diagnostic.json').read_text());best14=pair_rank14.iloc[0];ref14=summary[(summary.seed==43)&(summary.name=='v10_beta0.0')].iloc[0]
report14='# v14 — train 내부 오류 분석과 제출 후보 검증\n\n**결론: 제출 추천 후보 없음. 기존 v10을 유지하고 하루 한 번의 제출 기회를 진단용으로 쓰지 않습니다.** test/sample을 읽지 않았고 기존 홀드아웃의 예측·라벨로 점수를 계산하거나 후보를 선택하지 않았습니다. 이전 실행 메타데이터에서는 train 해시만 대조했습니다. 이 버전에는 새로운 홀드아웃 성능이나 Public 점수를 붙이지 않습니다.\n\n'
report14+='## 오류에서 출발한 가설\n\n개발 OOF에서 GBMLGG→LGG 153건, KIPAN→KIRC 140건, BRCA→OV 77건 등 쌍별 혼동이 컸습니다. THYM/PRAD/PCPG의 낮은 정밀도는 불균형 보정의 과도한 영향 가능성을, KIPAN의 낮은 재현율은 관련 클래스 간 조건부 구분의 어려움을 시사합니다. 이것은 관찰에 근거한 가설이지 임상 원인 확정이 아닙니다.\n\n'
report14+='## 단계 A: 학습 암종 빈도에 따른 작은 확률 보정\n\n각 fold 학습 환자 수에 1을 더한 prior의 beta승을 확률에 곱하고 정규화했습니다. beta 0/0.05/0.1/0.15/0.2/0.3/0.4, v6/v10 기준의 14개 계획입니다. 검증 정답으로 별도 계수를 학습하지 않았고, 예측 개수를 학습 비율에 강제로 맞추지 않았습니다. 모든 환자에게 같은 클래스별 계수를 적용하며 환자별 게이팅은 없습니다.\n\n'+rank[['name','clean','noise','score']].to_markdown(index=False)+'\n\n비보정 v10을 넘는 1차 후보가 없었습니다. 일부 재분할에서의 상승만 골라 채택하지 않았습니다.\n\n'
report14+='## 단계 B: 쌍별 조건부 분류\n\n사전 고정한 네 쌍, empirical/balanced LR(C=0.1)·XGBoost(depth2,150회)·LightGBM(3 leaves,120회), scope 2/4, alpha 0.25/0.5의 16개 계획을 비교했습니다. 쌍 학습 환자만으로 raw 유전자·변이·위치·유형·아미노산 변화·숫자·경로 어휘를 만들고 빈도 3 및 chi2 상위 512개를 선별했습니다. 선별·MaxAbs 스케일은 해당 fold의 쌍 학습 부분에서만 적합했습니다. 바깥 검증을 조기 종료에 사용하지 않았습니다. 각 쌍의 확률 합을 보존하고 조건부 확률만 고정 alpha로 혼합했습니다. 다른 클래스 확률은 그대로입니다.\n\n'+pair_rank14.to_markdown(index=False)+'\n\n'
report14+=f"최고 원본 F1 {best14.clean:.6f}, v10 {ref14.clean:.6f}, 차이 {best14.clean-ref14.clean:+.6f}. 사전 최소 개선 +0.004를 통과하지 못했습니다. stress F1은 {best14.noise:.6f} 대 v10 {ref14.noise:.6f}입니다.\n\n"
report14+='## 사후 진단: 후보 고정 후 여러 분할\n\n최고 후보를 튜닝 없이 고정해 seed 109/211/313에 적용했습니다. 이는 1차 채택 기준을 통과시킬 수 없는 별도 진단입니다. 같은 개발 환자를 재분할하므로 새로운 독립 환자 시험이 아닙니다. seed 313의 v10 ComplementNB는 그 fold의 학습 환자로 새로 학습했습니다. 이미 저장된 v6 baseline 313은 train 해시·ID·클래스·재구성 fold별 점수를 검증한 뒤 재사용했습니다. seed 313에는 기존 stress baseline이 없어 원본만 평가했으며 값을 만들어 넣지 않았습니다.\n\n'+paired_summary14.to_markdown()+'\n\n'
report14+='## 불확실성\n\n'+json.dumps(ci14,ensure_ascii=False,indent=2)+'\n\n이 CI는 고정된 개발 OOF에 대한 원본 프로필 그룹 단위 paired bootstrap 500회입니다. pooled Macro F1 변화에 대한 조건부 진단으로, fold 평균 변화와 다릅니다. 모델 재학습 변동과 후보 선택 편향을 교정하지 않으며 독립 최종 성능의 보증이 아닙니다.\n\n'
report14+='## 분리·재현·보관\n\n1차와 기존 재분할 모두 원본/정규화 프로필이 학습·검증 사이에 겹치지 않음을 확인했습니다. ID는 모델 입력에서 제외했고 test/sample open을 audit hook으로 차단했습니다. 기존 홀드아웃은 점수 계산에 사용하지 않았습니다. 과거 버전에서 test 분포와 홀드아웃을 관측했던 설계 편향은 사라지지 않습니다. 기준을 통과한 후보가 없어 전체 train 최종 모델이나 제출 CSV를 새로 만들지 않았습니다. 실험용 쌍 분류기의 학습·선별·검증은 이 노트북 안에서 수행합니다. 검증된 캐시가 없으면 baseline도 재학습하므로 별도 프로젝트 .py 파일은 필요하지 않습니다.\n'
(WORK/'analysis_report_v14.md').write_text(report14)
(WORK/'selected_variables_v14.md').write_text('# v14 선별 기준과 최종 변수 상태\n\n최종 제출 후보를 채택하지 않았으므로 새 전체-train 최종 변수 목록·모델은 없습니다. v10의 기존 목록은 selected_variables_v10.md를 참조합니다.\n\n실험용 이진 head는 쌍 학습 부분의 빈도 3 이상 raw 유전자/variant/position/type/aa/numeric/module 표현을 쓰고, binary chi2 상위 512개를 해당 fold 안에서 고릅니다. 각 fold의 목록은 달라집니다. LR만 해당 학습 행렬로 MaxAbs 스케일을 적합하며 tree는 원본 선택 행렬을 사용합니다. class_weight balanced 대 empirical은 목적 비교이며 인과·생물학적 유전자 역할을 확정하지 않습니다. 확률 빈도 보정은 fold 학습 라벨 수에만 의존하고 test나 예측 class 개수에는 의존하지 않습니다. 환자별 혼합 비중은 사용하지 않았습니다.\n\n개별 pair/알고리즘/fold별 환자·변수 수: artifacts/v14/pair_feature_audit.csv. 모든 파라미터는 protocol.json, pair_protocol.json과 노트북에 있습니다.\n')
proof14={'status':'passed','version':14,'accepted_new_candidate':False,'test_opened':False,'sample_opened':False,'holdout_scored':False,'daily_submission_used':False,'forbidden_attempts':attempts,'train_sha256':hashlib.sha256((DATA/'train.csv').read_bytes()).hexdigest(),'primary_v10_f1':float(ref14.clean),'best_pair_primary_f1':float(best14.clean),'primary_min_gain':.004,'diagnostic_candidate':diagnostic14,'diagnostic_summary':paired_summary14.reset_index().to_dict('records'),'bootstrap':ci14,'public_v10_user_reported':.3493803489,'source_bias_disclosed':True}
assert not attempts
(OUT/'verification.json').write_text(json.dumps(proof14,indent=2));print(json.dumps(proof14,indent=2),flush=True)


{
  "status": "passed",
  "version": 14,
  "accepted_new_candidate": false,
  "test_opened": false,
  "sample_opened": false,
  "holdout_scored": false,
  "daily_submission_used": false,
  "forbidden_attempts": [],
  "train_sha256": "92418b8441d058cfc68e939dd88725610750be4bc8edc51253cffc72fc4fc0ab",
  "primary_v10_f1": 0.442404469192675,
  "best_pair_primary_f1": 0.44464198354273005,
  "primary_min_gain": 0.004,
  "diagnostic_candidate": {
    "name": "lr_empirical_scope2_a0.5",
    "algo": "lr_empirical",
    "scope": 2,
    "alpha": 0.5,
    "clean": 0.44464198354273005,
    "noise": 0.34378519808371405,
    "duplicate": 0.44254950764002404,
    "score": 0.41417570031475465
  },
  "diagnostic_summary": [
    {
      "seed": 109,
      "view": "clean",
      "v10_f1": 0.43653660121133986,
      "candidate_f1": 0.4379607015301669,
      "delta_f1": 0.001424100318827078
    },
    {
      "seed": 109,
      "view": "duplicate",
      "v10_f1": 0.43930936851958896,
      "candidate_f1": 